In [52]:
import numpy as np
import pandas as pd
import neurokit2 as nk

Imported reshape function as it will be needed to make row-wise descriptives

In [ ]:
#Now we make the reshape function
#4 parameters, your dataframe, your window size or how many hz of rows should be in each row, purity for how much each row can tolerate label confounsion, and stride size for how much we should overlap
#By default this is every 10 seconds of data
def reshape_function(df, name = 'eda', window_size = 60, stride_size = 15, hertz=4, purity_threshold=0.8):
    window_row = window_size*hertz
    stride_row = stride_size*hertz
    #Get the subject names
    subjects = sorted(
        list(set(col.split("_")[0] for col in df.columns if "_" in col))
    )
    #Store the chunks of data while we're looping across the time span
    all_window = []
    for i in subjects:
        signal_col = f'{i}_{name}'
        label_col = f'{i}_Label'
        #Kill NAs in individual sequence
        individual_df = df[[signal_col,label_col]].dropna()
        total_rows = len(individual_df)
        #For each subject we take a chunk of their data by our determined window size then stack onto a list (in this case "all window")
        #Chunk is determined through standard range check then we check the purity threshold if we keep the chunk, based on birken's paper design
        for j in range(0,total_rows-window_row+1, stride_row):
            end = j + window_row
            signal_window = individual_df[signal_col].iloc[j:end].values
            label_window = individual_df[label_col].iloc[j:end].values

            # Determine the majority label and calculate if there exists a supermajority in the chunk to determine if kept
            value_counts = pd.Series(label_window).value_counts()
            majority_label = value_counts.index[0]
            majority_count = value_counts.iloc[0]

            purity = majority_count / window_row

            # Retain only windows meeting or exceeding the requested purity ratio
            if purity >= purity_threshold:
                row_data = [i, majority_label] + list(signal_window)
                all_window.append(row_data)

    feature_cols = [f"sample_{i}" for i in range(window_row)]
    columns = ["subject_id", "label"] + feature_cols

    return pd.DataFrame(all_window, columns=columns)

    





I also want visualization boxplots for each feature set similar to Zhu lili's breakdown

Import tonic and phasic files

In [54]:
tonic = pd.read_csv("../Data/wesad_eda/tonic_signal.csv")
phasic = pd.read_csv("../Data/wesad_eda/phasic_signal.csv")

This section is just applying the reshape function and verifying the outputs

In [55]:
tonic_10 = reshape_function(tonic, name='Tonic')

In [56]:
tonic_10

,subject_id,label,sample_0,sample_1,sample_2,sample_3,sample_4,sample_5,sample_6,sample_7,...,sample_230,sample_231,sample_232,sample_233,sample_234,sample_235,sample_236,sample_237,sample_238,sample_239
0,S10,0.0,-0.000006,0.000087,0.000607,0.001539,0.002864,0.004565,0.006626,0.009028,...,0.236105,0.236173,0.236259,0.236366,0.236493,0.236643,0.236815,0.237012,0.237234,0.237482
1,S10,0.0,0.237757,0.238061,0.238392,0.238747,0.239127,0.239528,0.239949,0.240388,...,0.236270,0.236243,0.236214,0.236182,0.236147,0.236107,0.236063,0.236015,0.235961,0.235901
2,S10,1.0,0.235834,0.235761,0.235682,0.235596,0.235504,0.235407,0.235304,0.235197,...,0.257519,0.257797,0.258075,0.258353,0.258630,0.258907,0.259184,0.259460,0.259735,0.260009
3,S10,1.0,0.260282,0.260554,0.260825,0.261095,0.261363,0.261630,0.261895,0.262159,...,0.259958,0.259969,0.259983,0.260000,0.260019,0.260040,0.260065,0.260091,0.260121,0.260154
4,S10,1.0,0.260189,0.260227,0.260267,0.260311,0.260356,0.260404,0.260454,0.260506,...,0.270468,0.270685,0.270889,0.271080,0.271257,0.271420,0.271568,0.271700,0.271816,0.271915
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1180,S9,4.0,0.537199,0.537046,0.536893,0.536741,0.536588,0.536436,0.536284,0.536133,...,0.494044,0.493961,0.493939,0.493983,0.494094,0.494277,0.494533,0.494867,0.495281,0.495778
1181,S9,4.0,0.496362,0.497034,0.497793,0.498634,0.499552,0.500545,0.501609,0.502740,...,0.554125,0.554033,0.553940,0.553845,0.553749,0.553651,0.553552,0.553451,0.553348,0.553243
1182,S9,4.0,0.553137,0.553029,0.552919,0.552807,0.552693,0.552579,0.552462,0.552345,...,0.530811,0.530725,0.530640,0.530555,0.530471,0.530387,0.530304,0.530221,0.530139,0.530057
1183,S9,0.0,0.558040,0.558471,0.558711,0.558768,0.558647,0.558355,0.557898,0.557282,...,0.633264,0.633266,0.633273,0.633282,0.633297,0.633316,0.633340,0.633370,0.633406,0.633449


Tonic descriptives row-wise for max and mean

In [57]:
sample_cols = [col for col in tonic_10.columns if col.startswith('sample_')]
tonic_descriptives = pd.DataFrame()
#Bring Labels into descriptives to the databas
tonic_descriptives['label'] = tonic_10['label']
#Bring subject ID over to descriptives
if 'subject_id' in tonic_10.columns:
    tonic_descriptives['subject_id'] = tonic_10['subject_id']
#Descriptives
tonic_descriptives['tonic_mean'] = tonic_10[sample_cols].mean(axis=1)
tonic_descriptives['tonic_max']  = tonic_10[sample_cols].max(axis=1)

tonic_descriptives

,label,subject_id,tonic_mean,tonic_max
0,0.0,S10,0.199474,0.237715
1,0.0,S10,0.245143,0.254486
2,1.0,S10,0.237035,0.260009
3,1.0,S10,0.264864,0.271328
4,1.0,S10,0.261337,0.271915
...,...,...,...,...
1180,4.0,S9,0.519602,0.537199
1181,4.0,S9,0.554567,0.568637
1182,4.0,S9,0.541264,0.553137
1183,0.0,S9,0.570054,0.633449


In [58]:
phasic_10 = reshape_function(phasic, name='Phasic')

Phasic descriptives row-wise for mean, peak counts and rise time standard deviation

In [59]:
sample_cols = [col for col in phasic_10.columns if col.startswith('sample_')]
phasic_descriptives = pd.DataFrame()
#Bring Labels into descriptives to the database
phasic_descriptives['label'] = phasic_10['label']
#Bring subject ID over to descriptives
if 'subject_id' in phasic_10.columns:
    phasic_descriptives['subject_id'] = phasic_10['subject_id']
#Descriptives
phasic_descriptives['phasic_mean'] = phasic_10[sample_cols].mean(axis=1)
phasic_descriptives['peaks_count'] = (phasic_10[sample_cols] > 0.01).sum(axis=1)
phasic_descriptives

,label,subject_id,phasic_mean,peaks_count
0,0.0,S10,1.742404e-02,29
1,0.0,S10,5.526673e-03,27
2,1.0,S10,3.993330e-03,15
3,1.0,S10,5.093199e-04,6
4,1.0,S10,5.298547e-04,4
...,...,...,...,...
1180,4.0,S9,1.309573e-03,11
1181,4.0,S9,3.425772e-03,8
1182,4.0,S9,2.829089e-07,0
1183,0.0,S9,2.517610e-02,50


Test example before I implemented on full dataframe the rise time standard deviation descriptive

In [60]:
row_example = phasic_10.iloc[0:1,2:]
signals, info = nk.eda_peaks(np.array(row_example).flatten(), sampling_rate=4)
# Calculate rise times in seconds
rise_times_sec = (info["SCR_Peaks"] - info["SCR_Onsets"]) / 4
np.nanstd(rise_times_sec)


0.5499719409228703

In [61]:
def rise_time_std(row, sampling_rate = 4):
    try:
        #Uses neurokits peak detection to get the onsets then convert it to a time scale and calculate the standard deviation
        signals, info = nk.eda_peaks(row, sampling_rate=sampling_rate)
        rise_times = (info["SCR_Peaks"] - info["SCR_Onsets"]) / sampling_rate
        return np.nanstd(rise_times)
    #There is a lot of rows with no peaks at all, need to decide how to handle the NAs where the std depreciates thereof
    except Exception:
        return np.nan
#Create a new column after apply row wise the above function
phasic_descriptives['phasic_std_rise_time'] = phasic_10[sample_cols].apply(
    lambda row: rise_time_std(row), axis=1
)

phasic_descriptives

,label,subject_id,phasic_mean,peaks_count,phasic_std_rise_time
0,0.0,S10,1.742404e-02,29,0.549972
1,0.0,S10,5.526673e-03,27,0.955317
2,1.0,S10,3.993330e-03,15,1.192828
3,1.0,S10,5.093199e-04,6,0.512596
4,1.0,S10,5.298547e-04,4,2.164486
...,...,...,...,...,...
1180,4.0,S9,1.309573e-03,11,0.471405
1181,4.0,S9,3.425772e-03,8,0.750000
1182,4.0,S9,2.829089e-07,0,0.257539
1183,0.0,S9,2.517610e-02,50,0.345482


Create and save file descriptive file

In [62]:
feature_dataframe = pd.concat(
    [phasic_descriptives, tonic_descriptives.drop(columns=["subject_id", "label"])],
    axis=1,
)
feature_dataframe

,label,subject_id,phasic_mean,peaks_count,phasic_std_rise_time,tonic_mean,tonic_max
0,0.0,S10,1.742404e-02,29,0.549972,0.199474,0.237715
1,0.0,S10,5.526673e-03,27,0.955317,0.245143,0.254486
2,1.0,S10,3.993330e-03,15,1.192828,0.237035,0.260009
3,1.0,S10,5.093199e-04,6,0.512596,0.264864,0.271328
4,1.0,S10,5.298547e-04,4,2.164486,0.261337,0.271915
...,...,...,...,...,...,...,...
1180,4.0,S9,1.309573e-03,11,0.471405,0.519602,0.537199
1181,4.0,S9,3.425772e-03,8,0.750000,0.554567,0.568637
1182,4.0,S9,2.829089e-07,0,0.257539,0.541264,0.553137
1183,0.0,S9,2.517610e-02,50,0.345482,0.570054,0.633449


In [63]:
feature_dataframe.to_csv("../Data/wesad_eda/feature_dataframe.csv", index=False)
